# Глава 4: Создание GPT-подобной модели для генерации текста с нуля

In [1]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install matplotlib torch tiktoken

Note: you may need to restart the kernel to use updated packages.


In [3]:
from importlib.metadata import version

print("Версия matplotlib:", version("matplotlib"))
print("Версия torch:", version("torch"))
print("Версия tiktoken:", version("tiktoken"))

Версия matplotlib: 3.10.9
Версия torch: 2.11.0
Версия tiktoken: 0.12.0


- Внедрение архитектуры LLM, подобную GPT

<img src="https://camo.githubusercontent.com/b54d2f0c11a8979798a453d6a47b3cc58c8ec44c4a5983074a0c895416d3b4ce/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830345f636f6d707265737365642f30312e77656270" width="800px">

## 4.1. Программирование архитектуры LLM

- Ранее обсуждались такие модели, как GPT и Llama, которые генерируют слова последовательно и основаны на декодирующей части оригинальной архитектуры transformer
- Поэтому эти LLM часто называют "подобными декодеру" LLM
- По сравнению с обычными моделями глубокого обучения, LLM имеют больший размер, в основном из-за огромного количества параметров, а не из-за объема кода
- Мы увидим, что многие элементы повторяются в архитектуре LLM

<img src="https://camo.githubusercontent.com/1a8b7609d44460e0b494557ab8f1a9de60283f9f66eb14bf9fa71356758aa65a/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830345f636f6d707265737365642f30322e77656270" width="800px">

- Ранее мы использовали небольшие размеры встраивания для ввода и вывода токенов для простоты иллюстрации, чтобы они уместились на одной странице
- В этой главе мы рассмотрим встраивание и размеры модели, аналогичные небольшой модели GPT-2
- Мы специально разработаем архитектуру самой маленькой модели GPT-2 (124 миллиона параметров), как описано в работе Рэдфорда и др. [Языковые модели являются многозадачными без контроля Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf ) (обратите внимание, что в первоначальном отчете это было указано как 117 миллионов параметров, но позже это было исправлено в модели хранилище весов)
- В будущем будет показано, как загрузить предварительно подготовленные веса в нашу реализацию, которая будет совместима с моделями типоразмеров с 345, 762 и 1542 миллионами параметров.

- Подробные сведения о конфигурации модели GPT-2 со 124 миллионами параметров включают:

In [4]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Размер словаря
    "context_length": 1024, # Длина контекста
    "emb_dim": 768,         # Размерность вложения
    "n_heads": 12,          # Количество целей внимания
    "n_layers": 12,         # Количество слоев
    "drop_rate": 0.1,       # Процент отсева
    "qkv_bias": False       # Смещение запроса-ключа-значения
}

- Мы используем короткие имена переменных, чтобы избежать использования длинных строк кода в дальнейшем
- `vocab_size` указывает на размер словаря в 50 257 слов, поддерживаемый токенизатором BPE.
- `context_length` представляет максимальное количество входных токенов модели, которое обеспечивается позиционными встраиваниями.
- `emb_dim` - это размер встраивания для входных токенов, преобразующий каждый входной токен в 768-мерный вектор.
- `n_heads` - это количество головок для управления вниманием в механизме управления несколькими головами.
- `n_layers` - это количество блоков-трансформаторов в модели, которые мы будем реализовывать в следующих разделах.
- `drop_rate` - это интенсивность механизма отсева, обсуждаемая в главе 3; 0,1 означает снижение 10% скрытых единиц во время обучения для уменьшения переобучения.
- `qkv_bias` определяет, должны ли "линейные" уровни в механизме многопоточного внимания включать вектор смещения при вычислении тензоров запроса (Q), ключа (K) и значения (V); мы отключим.

<img src="https://camo.githubusercontent.com/3d1152e718c0f7c8eaf876f03d500f089135acced7b2efe10d8a2292e24d3915/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830345f636f6d707265737365642f30332e77656270" width="800px">

In [5]:
import torch
import torch.nn as nn


class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # Использование заполнителя для блока-трансформера
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        # Использование заполнителя для LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Простой заполнитель

    def forward(self, x):
        # Этот блок ничего не делает и просто возвращает свои входные данные
        return x


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # Приведенные здесь параметры предназначены только для имитации интерфейса LayerNorm

    def forward(self, x):
        # Этот слой ничего не делает и просто возвращает свои входные данные
        return x

 ---

### Описание кода выше

Это **упрощённая (заглушка) GPT-подобная модель**. Она имитирует архитектуру GPT, но без реальных вычислений — все блоки просто возвращают входные данные без изменений. Нужна для отладки, тестирования и понимания структуры.

### 1. Главный класс `DummyGPTModel`

```python
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )
```

**Послойно:**

| Компонент | Что делает |
|-----------|------------|
| `tok_emb` | Таблица эмбеддингов токенов. Каждому ID токена сопоставляет вектор размером `emb_dim` |
| `pos_emb` | Таблица позиционных эмбеддингов. Каждой позиции в последовательности сопоставляет вектор |
| `drop_emb` | Dropout — случайно зануляет часть значений при обучении для борьбы с переобучением |
| `trf_blocks` | Цепочка из `n_layers` блоков-трансформеров (сейчас это заглушки) |
| `final_norm` | Финальная нормализация (заглушка) |
| `out_head` | Линейный слой, который превращает векторы размером `emb_dim` в логиты для каждого токена словаря |

### 2. Метод `forward` — прямой проход

```python
def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape  # например (2, 10) — 2 текста по 10 токенов
    
    tok_embeds = self.tok_emb(in_idx)   # токены → векторы
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))  # позиции → векторы
    x = tok_embeds + pos_embeds          # складываем: "что" + "где"
    x = self.drop_emb(x)                 # дропаут - это когда во время обучения нейросеть случайно затыкает часть нейронов, чтобы она не читерила и не запоминала тупо наизусть
    x = self.trf_blocks(x)               # трансформерные блоки (пока ничего не делают)
    x = self.final_norm(x)               # нормализация (пока ничего не делает)
    logits = self.out_head(x)            # векторы → вероятности токенов
    return logits
```

**Ключевая идея:** токен получает смысл из двух источников:
- **токен-эмбеддинг** — что это за слово
- **позиционный эмбеддинг** — на каком месте стоит

Они складываются, и дальше идут через трансформер.

### 3. Заглушки `DummyTransformerBlock` и `DummyLayerNorm`

```python
class DummyTransformerBlock(nn.Module):
    def forward(self, x):
        return x  # просто возвращает вход, ничего не меняя

class DummyLayerNorm(nn.Module):
    def forward(self, x):
        return x  # просто возвращает вход, ничего не меняя
```

Они нужны чтобы:
- код запускался без ошибок
- соблюдалась правильная структура модели
- позже их можно заменить на настоящие блоки

### Простыми словами:

Модель получает на вход индексы токенов → превращает в векторы → добавляет информацию о позиции → прогоняет через пустые блоки (как по трубе) → на выходе выдаёт логиты (оценки для каждого возможного токена). Это скелет GPT, в который потом вставят настоящие "мозги".

---

<img src="https://camo.githubusercontent.com/16436bc692fd2313bd5bc4d1cf001c0af6ee0f534875014765805a2bf8e62931/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830345f636f6d707265737365642f30342e776562703f313233" width="800px">

In [6]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [7]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Форма выходных данных:", logits.shape)
print(logits)

Форма выходных данных: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)
